# Tail Dependence in PV Ramp Events — Full Pipeline

This notebook runs the complete pipeline end-to-end by calling each module script in the correct order.  
Each cell executes one script and is preceded by a markdown cell explaining the step's purpose, inputs, and outputs.

**Dataset:** Utrecht PV Systems — 175 rooftop PV systems, 1-minute resolution, 2014–2017  
**Zenodo DOI:** [10.5281/zenodo.6906504](https://doi.org/10.5281/zenodo.6906504)

---
## Prerequisites
- Python environment with all packages from `requirements.txt` installed
- Run from the **project root** (the directory containing `src/`, `data/`, `A0_config.py`, etc.)
- Internet access for the initial data download (B1)

```bash
pip install -r requirements.txt
```

In [ ]:
import subprocess, sys
from pathlib import Path

# Ensure we run from the project root regardless of where the notebook was launched
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

def run(script: str) -> None:
    """Execute a pipeline script and stream its output to the notebook."""
    path = ROOT / script
    print(f"\n{'═'*60}")
    print(f"Running: {script}")
    print(f"{'═'*60}")
    result = subprocess.run(
        [sys.executable, str(path)],
        cwd=str(ROOT),
        capture_output=False,   # stream stdout live
    )
    if result.returncode != 0:
        raise RuntimeError(f"{script} exited with code {result.returncode}")

---
## Block A — Foundation

These steps validate the project setup before touching any data.  
They are fast (<5 s each) and should always pass before proceeding.

### A0 — Configuration validation

Reads `src/config.py` and asserts internal consistency across all parameter groups  
(seed, Utrecht metadata, ramp detection grid, Gate G0/G1 thresholds, synthetic test tolerances).  
Fails fast if any parameter is out of range or contradicts another.

**Output:** human-readable summary printed to stdout — no files written.

In [ ]:
run("A0_config.py")

### A1 — Directory structure

Creates all project directories declared in `cfg.DIRS` (data/raw, data/interim,  
data/processed, results/gates, results/figures, etc.).  
Idempotent — safe to re-run; existing directories are not touched.  
Places `.gitkeep` in new empty directories so they are tracked by git.

**Output:** directory tree under `data/` and `results/`.

In [ ]:
run("A1_setup_dirs.py")

### A2 — Environment check

Verifies that every Python package listed in `requirements.txt` is installed at an acceptable version.  
Also tests `pvlib` (clearsky model smoke test), the global random seed for reproducibility,  
and — if `rpy2` is present — that the R packages `texmex`, `evd`, and `POT` are importable.

**Output:** pass/fail report to stdout — no files written.

In [ ]:
run("A2_env_check.py")

### A3 — Synthetic validation tests

Runs four unit tests that verify the core statistical estimators **before** they are applied to real data:

| Test | What it checks |
|------|----------------|
| T1 | `chi_hat` ≈ 0 for two independent AR(1) series |
| T2 | `chi_hat` ≈ theoretical χ for a Gumbel copula with known θ |
| T3 | `chi_hat` cross-checked against the R package `texmex` (skipped if rpy2 absent) |
| T4 | `monte_carlo_ru()` matches the analytical relative uncertainty for a known geometry |

**Output:** PASS/SKIP/FAIL per test — no files written.

In [ ]:
run("A3_synthetic_tests.py")

---
## Block B — Data ingestion and pre-processing

These steps download the Utrecht dataset, apply quality control, compute the clearsky index,  
detect ramp events, and produce the final analysis-ready datasets.

> **Note:** B1 downloads ~2.9 GB from Zenodo. Steps B2–B8 require B1 to have completed successfully.  
> The download is resumable — re-running B1 will continue from where it stopped.

### B1 — Download Utrecht dataset

Downloads three files from Zenodo (DOI 10.5281/zenodo.6906504):
- `metadata.csv` — station IDs, bounding-box coordinates, DC/AC capacity, tilt, azimuth
- `qcpv.py` — the quality-control routine published alongside the dataset
- `filtered_pv_power_measurements_ac.csv` (~2.9 GB) — 175-column wide-format power time series, already filtered by the authors' qcpv routine

Then builds `coords.parquet`: centroid lat/lon for each station derived from the bounding box, plus capacity in both W and kWp.

**Output:**
- `data/raw/utrecht/` — raw downloaded files
- `data/interim/coords.parquet` — 175 rows × 12 columns

In [ ]:
run("B1_download_utrecht.py")

### B2 — Quality control and censorship report

Loads the pre-filtered power data (already processed by Lanzilao & Meyer's `qcpv` routine)  
and clips each station's series to its declared operational window `[begin_ts, end_ts]`.  
Produces a per-station censorship report documenting what fraction of timestamps is valid —  
essential for understanding measurement gaps before any statistical inference.

**Output:**
- `data/interim/power_qc.parquet` — wide-format power (W), DatetimeIndex UTC
- `data/interim/qc_report.csv` — n_total, n_valid, pct_valid, pct_censored per station

In [ ]:
run("B2_qc.py")

### B3 — Normalize by DC capacity

Converts raw power measurements (W) to dimensionless normalized power:

$$p^*_i(t) = \frac{P_i(t) \; [\text{W}]}{\text{DC\_capacity}_i \; [\text{W}]}$$

Values outside $[0,\; 1.1]$ are set to NaN (the 10 % overirradiance headroom tolerates brief STC exceedances).  
This normalization puts all 175 systems on a common scale, enabling spatial comparison.

**Output:**
- `data/interim/power_norm.parquet` — $p^*_i(t)$, dimensionless
- `data/interim/norm_report.csv` — clipping counts per station

In [ ]:
run("B3_normalize.py")

### B4 — Clearsky index $k_i(t)$

Computes the **clearsky index** for each station using the Ineichen–Perez model (pvlib):

$$k_i(t) = \frac{p^*_i(t)}{p_{\text{cs},i}(t)}$$

where $p_{\text{cs},i}(t) = \text{GHI}_\text{clearsky}(t) \;/\; 1000 \text{ W/m}^2$.  
$k_i$ is undefined (NaN) at night ($p_{\text{cs}} < 0.005$) and clipped to $[0, 1.5]$.  
By removing the deterministic diurnal envelope, $k_i$ isolates cloud-driven variability —  
the quantity on which all subsequent extreme-value analysis is performed.

**Output:**
- `data/interim/clearsky_index.parquet` — $k_i(t)$, same shape as power_norm
- `data/interim/clearsky_report.csv` — median and P95 of $k_i$ per station

In [ ]:
run("B4_clearsky.py")

### B5 — Ramp event detection (true Swinging-Door Trending)

Applies the genuine **Swinging-Door Trending (SDT)** algorithm (Bristol, 1990 — the technique used in the canonical ramp-detection literature, e.g. Florita, Hodge & Orwig 2013) to $k_i(t)$ per station, in two phases:

1. **Compression:** a "door" of slope bounds narrows as each new point arrives; when it closes, the last compatible point is archived as a breakpoint. This yields a piecewise-linear approximation of $k_i(t)$ within tolerance $\varepsilon$ — segment boundaries are genuine trend-change points, not artifacts of an arbitrary fixed window.
2. **Ramp extraction:** a segment between two consecutive breakpoints is a ramp event if **both** $|\Delta k| \geq \Delta$ (magnitude) **and** duration $\geq$ `min_duration` (persistence) hold.

Compression is JIT-compiled via `numba` for speed (~9 s for 175 stations × 2.1 M timestamps).

Default parameters (from `cfg.RAMP`):
- $\varepsilon = 0.02$ (SDT compression tolerance, dimensionless)
- $\Delta = 0.10$ (dimensionless ramp magnitude threshold)
- `min_duration = 10 min` (persistence criterion)

Each detected event records `start_ts`, `end_ts`, `delta_k` (net change over the event), `direction` (up/down), and `duration_min` (variable, emergent from the compression — not a windowing artifact).  
Override defaults at the command line: `python B5_ramp_detection.py --epsilon 0.03 --delta 0.20 --min-duration 15`

> **Methodology evolution log (2026-07-01):**
> 1. **v1 bug:** initial implementation recorded one event per sliding-window step (fixed duration $=\tau$), causing massive double-counting → 146 events/day/station (implausible).
> 2. **v2 fix:** merged contiguous flagged runs into single variable-duration events → 23.9 events/day/station. Still a *windowed-threshold heuristic*, not true Swinging-Door.
> 3. **v3 — true SDT:** replaced the heuristic with the genuine SDT compression + magnitude threshold. Validated with 4 synthetic tests (pure noise → 0 events; isolated ramp → 1 event; two ramps → 2 non-overlapping events; night-time NaN gap never bridged). **Without a duration filter this produced 59.9 events/day/station, with 63 % of events lasting exactly 1 minute** — a symptom of capturing high-frequency cloud-edge variability, not sustained ramps.
> 4. **v3 final:** added a **minimum-duration criterion (10 min)** on top of the SDT magnitude threshold, following standard practice in the grid-integration ramp literature (ramp = magnitude **and** persistence). Result: **2.18 events/day/station**, median duration 14 min, P90 36 min, max 446 min — physically plausible and comparable to values reported in the literature. See `results/FINDINGS.md` for the full log.

**Output:**
- `data/interim/ramps.parquet` — one row per event
- `data/interim/ramp_report.csv` — event counts and daily rates per station

In [ ]:
run("B5_ramp_detection.py")

### B6 — Parameter sensitivity sweep

Evaluates ramp detection across a grid of $(\varepsilon, \Delta)$ combinations (`min_duration=10 min` held fixed, per the B5 finding) to check whether **at least 90 % of stations** accumulate ≥ 30 exceedances — the minimum required for a reliable GPD fit (Gate G1 threshold).

Grid (from `cfg.RAMP`):
- $\varepsilon \in [0.01, 0.02, 0.03, 0.05, 0.08]$ (SDT compression tolerance)
- $\Delta \in [0.10, 0.15, 0.20, 0.25, 0.30]$ (ramp magnitude threshold)

For efficiency, the SDT compression (which depends only on $\varepsilon$) is computed **once per $\varepsilon$** and reused across all $\Delta$ values, instead of recomputing the full JIT loop for every grid cell. Rate is normalized by **calendar days** (consistent with the B5 report metric).

> **Result (2026-07-01, after migrating to true SDT + duration filter):** unlike the pre-SDT grid  
> (where `frac_sufficient ≥ 0.99` for all 25 combinations, with no discriminating power), the new  
> grid shows **real discrimination**: `frac_sufficient` ranges from **0.47** ($\varepsilon=0.01, \Delta=0.30$ —  
> insufficient) to **0.99** (most combinations). The default pilot configuration  
> ($\varepsilon=0.02, \Delta=0.10$ → 2.18 ramps/day/station) sits comfortably in the sufficient region  
> (frac=0.99) and is retained. Final parameter justification still awaits: (1) visual inspection  
> (B5, pending) and (2) confirming Gate G1 conclusions (χ estimates) are stable across the grid.

**Output:**
- `data/interim/sensitivity_grid.csv` — 25-row results table → Table 6 of the paper
- `results/figures/B6_sensitivity_heatmap.png`

In [ ]:
run("B6_sensitivity.py")

### B5b — Visual validation of detected ramps

ROADMAP B.5 requires a mandatory visual check: overlay detected events on $k_i(t)$ to confirm (a) no false positives during flat periods and (b) obvious sharp transitions are captured.

Automatically selects the station closest to the network's median ramp activity and plots the 3 consecutive days with the most events, with green/red bands (up/down) overlaid on $k_i(t)$, plus a zoomed view of the most active day.

> **Result (2026-07-01):** confirmed — no spurious flags during flat/noisy-but-stable periods, and sharp transitions are correctly captured. **But** a large gradual rise (cloud clearing over ~4 hours) was **not** captured as a single event — the SDT fragments it into short sub-threshold segments. This is the known SDT limitation noted in B5, and motivated the follow-up empirical test below (F2).

**Output:**
- `results/figures/B5_visual_check.png` — 3-day overview
- `results/figures/B5_visual_check_zoom.png` — zoom on the most active day

In [ ]:
run("B5b_visual_check.py")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "results/figures/B5_visual_check.png")))
display(Image(filename=str(ROOT / "results/figures/B5_visual_check_zoom.png")))

### F2 — Spatial coherence: sharp ramps vs. gradual transitions

The visual check above raised a question: is the SDT detector's blindness to gradual, multi-hour transitions a **defect**, or does it reflect a genuinely **different physical process** (synoptic-scale frontal clearing vs. localized, wind-advected cloud edges)?

**Hypothesis:** if sharp SDT ramps are caused by localized cloud edges advected by wind, the onset **lag** between stations should **grow with inter-station distance**. If gradual transitions are caused by a synoptic-scale event (regional front), onset lag should be small and roughly **independent of distance** (spatially coherent).

**Method:**
1. $T_{sharp}$(station, event) — onset of each strong ("up", $|\Delta k| \geq 0.15$) SDT ramp, grouped into 90-minute global blocks so that only ramps plausibly belonging to the *same* cloud event across stations are compared (comparing a station's morning ramp to another's afternoon ramp would be meaningless).
2. $T_{gradual}$(station, day) — first sustained crossing of a 30-min rolling mean of $k_i(t)$ above 0.5, per station-day.
3. For each qualifying block/day (≥15 or ≥20 stations respectively), sample station pairs and compute $(\text{distance}_{km}, |\Delta T|_{min})$.
4. Compare the distance–lag EFFECT SIZE (r, slope) between the two groups — **not the p-value**: with up to 300 pairs drawn repeatedly from only ~174 stations, pairs are pseudo-replicated and nominal p-values collapse to ≈0 regardless of true effect size. As a robustness check, the correlation is recomputed using **one pair per block/day** (an approximately independent sample).

> **Result (2026-07-01):** sharp ramps show $r(\text{distance}, \text{lag}) = 0.21$, slope $=0.42\pm0.002$ min/km, median lag $=14$ min ($n=656{,}508$ pooled pairs; **robustness check with 1 pair/block, $n=2{,}637$: $r=0.207$** — essentially identical, confirming the effect is real, not a pseudo-replication artifact). Gradual transitions show $r = 0.047$ — **~4.5× weaker spatial coherence** — slope $=0.55\pm0.02$ min/km, median lag $=47$ min, more scattered (1 pair/day check, $n=1{,}213$: $r=0.050$).
>
> **Decision: scope delimited to sharp ramps, with physical justification.** Sharp ramps are the advection signature this study's hypothesis (RQ1–RQ3, spatial propagation and tail dependence) is designed to investigate — their lag grows with distance, consistent with localized cloud-edge features carried by wind across the array. Gradual, synoptic-scale clearing is a distinct physical process (near-simultaneous, largely distance-independent onset) that belongs to a different research question and is deliberately set aside as future work (a natural "Paper 2"), **not** a detector failure. Accordingly, the SDT compression tolerance ($\varepsilon$) is **not** being retuned and a multi-scale detector is **not** being implemented now — both would dilute this focused scope.

**Output:**
- `results/gates/f2_spatial_coherence.csv` — all sampled (distance, lag, type) pairs
- `results/figures/F2_lag_vs_distance.png`

In [ ]:
run("F2_ramp_spatial_coherence.py")

---
## Block C — Decision Gates

These scripts run the go/no-go gates that determine whether the project proceeds.  
Each gate produces a written decision record in `results/gates/` alongside figures and data.

### C0 — Gate G0: Spatial adequacy of geocoordinates

Answers: does the 150 m × 150 m anonymization grid in the Utrecht dataset introduce
unacceptable uncertainty in the pairwise distance matrix?

**Method (Monte Carlo position perturbation — ROADMAP F1):**
For each of 1 000 draws, every station receives a random offset drawn from
Uniform(−75 m, +75 m) in both X and Y. The pairwise haversine distance matrix is
recomputed for each draw. For every pair (i, j) the relative uncertainty is:

$$RU_{ij} = \frac{P_{97.5}(d_{ij}) - P_{2.5}(d_{ij})}{\text{median}(d_{ij})}$$

**Decision rule:**
- `fraction_flagged` = #{pairs with RU > 20%} / #{all pairs}  
- If `fraction_flagged < 5%` → **G0 APPROVED WITH CAVEAT** (flag affected pairs in F6)  
- Otherwise → **G0 PARTIAL FAILURE** (adopt errors-in-variables regression in F6)

**Result (2026-07-01):** G0 **APPROVED WITH CAVEAT** — 2.79 % of pairs flagged,
RU median = 0.020, RU P95 = 0.133.

**Output:**
- `results/gates/gate0_results.parquet` — (station_i, station_j, d_nominal_m, d_median_mc, RU, flagged)
- `results/gates/gate0_decision.md` — written decision with date
- `results/figures/gate0_ru_histogram.png`
- `results/figures/gate0_ru_vs_dist.png`

In [ ]:
run("C0_gate0.py")

### Viewing the findings log

Every gate and analysis phase writes to two files simultaneously:

| File | Purpose |
|---|---|
| `results/log/run_log.jsonl` | Machine-readable audit trail — one JSON per `(script, gate, phase)`, deduplicated so a re-run **replaces** the previous record (same semantics as `FINDINGS.md`), keeping both files as a faithful snapshot of the current state |
| `results/FINDINGS.md` | Human-readable cumulative narrative — ready to cite in the paper |

The cell below loads the latest run log and displays the findings as a table.

In [ ]:
import json
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(ROOT))
from src.config import cfg

log_path = cfg.DIRS["gates"].parent / "log" / "run_log.jsonl"

if log_path.exists():
    records = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
    summary = pd.DataFrame([{
        "timestamp": r["timestamp"][:19].replace("T", " "),
        "gate/phase": r["gate"] or r["phase"],
        "script":     r["script"],
        "decision":   r["decision"],
        "paper_ref":  r["paper_ref"],
    } for r in records])
    display(summary)
    print(f"\nTotal runs logged: {len(records)}")
    print(f"Full narrative: results/FINDINGS.md")
else:
    print("No runs logged yet.")

### B7 — Wind covariate join (real data, KNMI De Bilt, 10 m)

Associates wind speed and direction with each ramp event's timestamp, for use in the
anisotropy analysis (F6). Downloads **real** historical hourly wind from KNMI station
260 (De Bilt) automatically — **no registration needed** — via a public POST endpoint:

```
https://www.daggegevens.knmi.nl/klimatologie/uurgegevens
```

(Not to be confused with `opendata.knmi.nl/Actuele10min...`, which only covers the
last 10 days and is unusable for 2014-2017.)

**Two bugs found and fixed during development** (worth knowing if extending this script):
1. Building the output DataFrame from `pandas.Series` (not arrays) together with an
   explicit `index=` parameter silently triggers index-alignment reindexing — since the
   Series carried a `RangeIndex` unrelated to the target `DatetimeIndex`, every value
   became `NaN`.
2. The KNMI API's `HH` digits in `start`/`end` do **not** define a continuous time
   range — they restrict **which hours of the day** are returned across the *entire*
   period (e.g. `start=...07`, `end=...15` returns only hours 7-15 of *every* day).
   The first (buggy) run silently matched only 48.1% of ramp events; forcing `HH=01`
   / `HH=24` fixed this to 100%.

**Known limitation:** 10 m is surface wind, not necessarily representative of wind at
cloud-base height (typically 600-1500 m in the Dutch climate). Addressed as an optional
enhancement in **B7b** below (real height-resolved wind via CERRA).

**Output:**
- `data/interim/wind_joined.parquet` — ramps + wind_speed_ms + wind_dir_deg (real, 10m)
- `data/raw/wind/knmi_debilt_wind.csv` — cached raw download

In [ ]:
run("B7_wind_join.py")

### B7b — Wind covariate join, height-resolved (optional, CERRA)

Complements B7's 10 m surface wind with **height-resolved** wind (100 m, 200 m, 500 m)
via **CERRA** (Copernicus European Regional ReAnalysis, 5.5 km resolution) — chosen over
ERA5 because the Utrecht network (~44 x 50 km) spans only **~1.4 x 1.6 ERA5 grid cells**
(31 km resolution — essentially one point for the whole network), vs. **~8.0 x 9.2 CERRA
cells** (5.5 km) — a ~6x improvement, enough to resolve internal spatial structure if
ever needed (though the anisotropy test itself only needs one representative regional
wind time series, since synoptic wind direction is coherent well beyond the network's
extent). CERRA is analysis-only every **3 hours** (not hourly) — a real trade-off against
its finer spatial resolution, documented and handled with a wider merge tolerance (100
min). Combined with the 10 m KNMI wind, this gives a **4-height vertical profile**
(10/100/200/500 m); which height best explains any observed anisotropy in F6 should be
reported as an explicit finding, not assumed a priori.

**Requires a one-time, free CDS (Copernicus Climate Data Store) registration** — the
script detects if this is missing and prints step-by-step instructions, then **skips
gracefully** (does not block the rest of the pipeline):

#### Registration walkthrough (one-time, ~5 minutes)

1. **Create a free account** at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu/)
   (top-right "Login/Register" → "Create new account"). Standard email + password sign-up,
   no institutional affiliation or approval needed.
2. **Get your API token**: log in → click your username (top right) → **"Personal Access
   Token"** page → copy the token string shown there. This is your `key` for `~/.cdsapirc`
   below — treat it like a password (do not commit it to git).
3. **Accept the dataset licence** — a step that is easy to miss and **will cause a
   `403 required licences not accepted` error even with a valid token** if skipped:
   open the [CERRA height-levels dataset page](https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-height-levels),
   go to the **"Download"** tab, scroll to **"Terms of use"**, and click **accept**. This is
   a per-dataset acceptance, separate from account creation, and is a one-time action tied
   to your account (not needed again for future runs, even for other CERRA variables).
4. **Create `~/.cdsapirc`** (in your home directory, not the repo) with:
   ```
   url: https://cds.climate.copernicus.eu/api
   key: <YOUR_TOKEN_FROM_STEP_2>
   ```
5. **Install the client libraries** (already listed in `requirements.txt`):
   `pip install cdsapi xarray netcdf4` — note: if you have multiple Python installs (e.g.
   Anaconda + system Python), make sure you install into the **same interpreter** that runs
   this notebook/script (`python3 -m pip install ...` rather than a bare `pip install ...`
   avoids installing into the wrong environment).
6. Re-run this cell.

#### What happens during the actual download (learned from running it for real)

Requesting the full 2013-2018 period in one call is rejected by the CDS API with
`403 cost limits exceeded` (too many year×month×day×hour×height×variable fields in a
single request) — even a single full year is too large. The script therefore splits the
request into **24 quarterly chunks** (one per 3-month window, 2013-2018) and downloads
them one at a time. It is **idempotent**: if interrupted, re-running it resumes from the
first missing quarter rather than re-downloading everything.

Each quarterly request can sit in the CDS queue (`status: accepted` → `running` →
`successful`) anywhere from **a few minutes to about an hour**, depending on how busy the
CDS backend is — this is normal and not something the script or your connection controls.
With 24 quarters, expect the full download to take a while; it is safe to leave it running
in the background and check back later.

Two more CDS-specific quirks the script works around automatically (documented here for
transparency, no action needed from you):
- The CERRA native grid is a Lambert conformal conic projection; requesting an `area`
  crop without also requesting a regridded `grid` triggers a CDS server-side bug
  (`croppedRepresentation() not implemented for RegularGrid`). The script requests both
  `area` **and** `grid` together, which avoids the bug and also shrinks the download a lot.
- The wind-speed variable in the returned NetCDF is named `si100` regardless of which
  height you are reading (100/200/500 m all live inside that one variable, indexed by a
  `heightAboveGround` dimension) — the extraction code looks it up by prefix rather than
  assuming a per-height variable name.

**Output (once configured):**
- `data/raw/wind/cerra_by_year/cerra_<year>_<mm>-<mm>.nc` — cached raw quarterly downloads (24 files)
- `data/interim/wind_joined.parquet` — extended with `cerra_wind_speed_ms_{100,200,500}m` / `cerra_wind_dir_deg_{100,200,500}m`

In [ ]:
run("B7b_wind_cerra.py")

### B8 — Temporal train / test split

Assigns a `split` label to every timestamp and ramp event:

| Split | Period | Purpose |
|-------|--------|---------|
| `train` | 2014-01-01 → 2016-12-31 | Model fitting, GPD estimation, tail dependence analysis |
| `test`  | 2017-01-01 → 2017-12-31 | Out-of-sample validation |

The split is **purely temporal** (block split) — no shuffling — to prevent temporal leakage.  
The column is added in-place to `clearsky_index.parquet`, `ramps.parquet`, and `wind_joined.parquet`.

**Output:**
- `data/interim/clearsky_index.parquet` — updated with `split` column
- `data/interim/ramps_split.parquet` — ramps with `split` column
- `data/interim/split_report.csv` — event counts by split and station

In [ ]:
run("B8_temporal_split.py")

---
### C1 — Gate G1: Tail dependence diagnostic

Answers: do nearby PV stations exhibit upper-tail dependence in ramp magnitudes beyond
what chance would predict? This is the central go/no-go gate for the spatial pillar
of the study (RQ1–RQ3).

**Methodological design.** χ̂/χ̄ (Coles; Heffernan–Tawn; Ledford–Tawn) require paired
observations on a common time index, but ramps are an irregular point process per
station (2–4/day on average, up to 21 in a single day). The common index adopted here
is the **calendar day**: $M_i(d)$ = max $|\Delta k|$ among station $i$'s ramps on day
$d$ (0 if none), rank-transformed per station (training days only, 967 days × 174
stations). The E.2 coincidence window ($\Delta t_{ij} = \text{dist}_{ij} / 15\,\text{m/s}$,
capped at 30 min) does **not** filter the χ calculation — it is used as a
complementary physical check: among joint-exceedance days, what fraction have event
times consistent with cloud-front propagation?

**Steps (ROADMAP Block E):**
1. **E.1** Uniform margins per station (rank-based, train only) → `uniform_margins.parquet`
2. **E.2** Coincidence-window alignment check (physical corroboration, not gating)
3. **E.3** $\hat\chi$ and $\bar\chi$ for all 15,051 pairs, $u \in \{0.90, 0.95\}$ → `chi_estimates_raw.parquet`
4. **E.4** Moving block bootstrap (95% CI), pairs < 5 km only → `gate1_results.parquet`
5. **E.5** Block-permutation p-values + Benjamini–Hochberg FDR ($\alpha=0.05$) at $u=0.95$
6. **E.6** Decision: `fraction_significant > 0.30` → **G1 APPROVED**

**Result (2026-07-01):** **G1 APPROVED** — $\hat\chi(0.95)$ median = 0.124 among pairs
< 5 km (vs. 0.103 across all pairs), decaying qualitatively with distance. After block
bootstrap (100 draws, block length = 10 days, ACF-estimated) and BH-FDR correction,
**52.1% of close pairs are significant** (threshold: 30%).

**Honest caveat (E.2 physical check):** only **15.4%** of joint-exceedance days have
event times within the cloud-advection window (v = 15 m/s; 12.2% at 20 m/s). This does
**not** invalidate $\chi > 0$ (independence is still rejected, and $\chi$ decays with
distance as expected of a real spatial process), but it means the daily-block design
mixes "shared-regime" dependence with strict event-to-event co-occurrence. **F4** (GPD
marginals) can proceed as-is; **F5/F6** (Heffernan–Tawn, anisotropy) require refining
the pairing to event-level matching before proceeding — flagged in `ROADMAP.md`.

The estimator ($\hat\chi$, $\bar\chi$) is reused from `A3_synthetic_tests.py`, already
validated synthetically (independence → χ≈0; Gumbel copula θ=2 → χ≈0.586).

**Output:**
- `data/processed/uniform_margins.parquet`
- `results/gates/chi_estimates_raw.parquet` — χ̂/χ̄ for all 15,051 pairs
- `results/gates/gate1_results.parquet` — 3,070 close pairs with bootstrap CI, p-value, FDR flag
- `results/gates/gate1_decision.md` — written decision with date
- `results/figures/gate1_chi_vs_distance.png`
- `results/figures/gate1_ci_examples.png`

In [ ]:
run("C1_gate1.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/gate1_chi_vs_distance.png")))
display(Image(filename=str(ROOT / "results/figures/gate1_ci_examples.png")))

---
### C1b — Gate G1 event-level refinement

Investigates the Gate G1 caveat: the original design paired ramps by **daily block
maximum**, which mixes "shared-regime" dependence with strict event-to-event
co-occurrence. This script rebuilds the paired series at the **event level**: for each
close pair (< 5 km), an "extreme event" at station $i$ = a ramp with $|\Delta k|$ above
station $i$'s own 95th percentile (all training ramps, both directions). Each extreme
event is matched to the nearest-in-time ramp at the neighbouring station within the
physically-justified coincidence window, in both directions ($i\to j$ and $j\to i$).

Two matching rules are compared: **nearest-in-time** and **any-extreme-in-window**
(does the neighbour have *any* extreme event in the window, not just the closest one) —
a check against SDT fragmentation (B5b) diluting the nearest-in-time match.

$$\chi_{\text{event}}(u) := P(\text{matched event also extreme} \mid \text{extreme event, matched within window})$$

Under independence, $\chi_{\text{event}}(u) \to 1-u = 0.05$ (not zero) — this is the
correct baseline, not $\chi=0$.

**Result (2026-07-01):** the two matching rules gave nearly identical results (26,227 vs.
26,233 matches out of 736,342 extreme events evaluated) — fragmentation dilution is
**not** the dominant effect. But $\chi_{\text{event}}(0.95)$ pooled = **0.036**, at or
**below** the independence baseline (0.05) — while the raw coincidence rate (any
magnitude) averages **11.2%** per pair (11.5% pooled across all matched events, the
figure used in F5 below), far above the 1.0% expected under an independent-Poisson null
(+10.2 points excess).

**Important finding:** ramp *activity* co-occurs far more than chance would predict
(shared weather regime), but this does **not** translate into magnitude-conditional tail
dependence at the matched-event level. This refines (and sharpens) the original Gate G1
caveat: the daily-block $\chi > 0$ finding is predominantly a **shared-regime** effect,
not "same cloud edge, comparable magnitude, minutes apart" event coupling. **F5** should
not assume event-level pairing reproduces the daily-block dependence strength without
testing it explicitly; **F6** (anisotropy/propagation-speed regression) is weakened by
this finding.

**Output:**
- `data/processed/aligned_pairs.parquet` — 736,342 extreme-event evaluations with
  `partner_is_extreme` (nearest-in-time) and `partner_extreme_anywhere_in_window` flags
- `results/gates/event_pairing_summary.parquet` — per-pair χ_event, coincidence rate vs. null
- `results/gates/gate1_event_refinement.md` — written finding with date
- `results/figures/gate1b_chi_event_vs_daily.png`
- `results/figures/gate1b_coincidence_vs_null.png`

In [ ]:
run("C1b_event_pairing.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/gate1b_chi_event_vs_daily.png")))
display(Image(filename=str(ROOT / "results/figures/gate1b_coincidence_vs_null.png")))

---
### C2 — Gate G2: Marginal modelling (GPD with covariates)

Answers: are ramp-magnitude exceedances, per station and per direction (up/down), well
described by a Generalized Pareto Distribution with enough exceedances and stable
parameters? This gate conditions F5 (Heffernan–Tawn), F6 (spatial structure), F7 (return
levels) and F8 (the central RQ3 result).

**Design decisions** (confirmed with the user before coding):
- Separate GPD per **direction** (up/down) per station — 174×2 = 348 series (not pooled).
- **Adaptive per-series threshold** selection — automated version of the parameter
  stability plot: the smallest `u` from which subsequent ξ̂ estimates stay within the
  95% CI of ξ̂(u) (asymptotic $\text{Var}(\hat\xi)\approx(1-\xi)^2/n$, Smith 1985).
  Validated visually on 5 sample stations (low → high activity) before applying to all.
- **Automatic declustering**: run-length derived from the Ferro–Segers (2003) intervals
  estimator of the extremal index θ (no run-length chosen a priori) — searches for the
  run-length whose resulting cluster count best matches `N_exceedances × θ̂`.
- **Low-activity stations**: threshold automatically lowered until the minimum exceedance
  count is reached, flagged `low_confidence`.
- σ(t) = exp(β₀ + β₁·solar_elevation_std(t) + β₂·seasonality(t)); ξ constant per series.

**Result (2026-07-01):** **G2 APPROVED** — 341/348 series (98.0%) meet the criterion
(≥50 declustered exceedances, converged fit, |ξ̂|<0.5), after the robustness cross-check
below. Median θ̂ = 0.469 (moderate temporal clustering, consistent with the known SDT
fragmentation). Median ξ̂ = -0.143 (up: -0.160, down: -0.134) — near-exponential, mildly
bounded tail, physically consistent with a hard ceiling on clearsky-index ramps.

**Robustness cross-check (resolves the original caveat):** of the 10 series first
failing, only `ID117` (121–128 total ramps) fails from genuine data scarcity. The other
9 had adequate raw sample size (700–1,600 ramps) but implausible covariate-model shape
estimates (ξ̂ < -0.5). Each was refit with a plain (no-covariate) 2-parameter GPD on the
same declustered sample: **3 of 9 were rescued** (plain ξ̂ plausible, `covariate_model_
unstable=True`, β₁=β₂=0 not estimated) — confirms 4-parameter MLE instability, not a
pathological tail. **The remaining 6 stayed implausible even in the simplest possible
model** — a genuine finding, not an artifact: 5 of them (`ID015/024/025/040/051`) are
exclusively the **"up"** direction, consistent with a real physical ceiling (k_i cannot
exceed its clearsky-normalized maximum, so large upward excursions are mechanically
bounded in a way downward ramps are not); the 6th (`ID117`-down) is the same
low-activity station already flagged for data scarcity. All 6 remain correctly excluded
from `gate2_pass` and should be reported as a genuine direction-asymmetric
tail-boundedness finding.

**Output:**
- `results/gates/gpd_marginal_params.parquet` — GPD parameters per station × direction (Table 1)
- `results/gates/gate2_decision.md` — written decision with date
- `results/figures/gate2_threshold_diagnostics.png` — Figure 1 (MRL + stability plots, 5 sample stations)
- `results/figures/gate2_qq_plots.png` — Figure 2 (GPD QQ-plots, same stations)
- `results/figures/gate2_extremal_index.png` — Figure 7 (θ distribution)

In [ ]:
run("C2_gate2.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/gate2_threshold_diagnostics.png")))
display(Image(filename=str(ROOT / "results/figures/gate2_qq_plots.png")))
display(Image(filename=str(ROOT / "results/figures/gate2_extremal_index.png")))

---
### F5 (pilot) — Heffernan–Tawn conditional-extremes feasibility

Motivated by the C1b finding (chi_event at the independence baseline), this pilot tests
**directly** whether there is enough signal to justify the full Heffernan–Tawn fit (flexible
$\beta$ + Keef–Papastathopoulos–Tawn 2013 constraints) before investing in it. Uses a
deliberately simplified $\beta=0$ model ($Y_j = \alpha X_i + Z$), for which $\hat\alpha$
reduces to an intercept-free OLS slope on standard-Laplace margins, applied to
`aligned_pairs.parquet` (unmatched extreme events get $Y_j=0$).

**Artifact discovered and documented:** the raw $\hat\alpha$ came out **outside the
plausible Heffernan–Tawn range [-1,1]** for all 6,132 directional pairs tested (median
-1.91) — this is **not** genuine negative dependence, but an artifact of the "no match →
magnitude 0" substitution (88.5% of cases): mapped through each station's ECDF, magnitude=0
falls below every observed ramp, creating a near-constant, strongly negative Laplace-scale
floor that destabilizes the OLS estimator.

**Reliable evidence (artifact-free):**
1. Despite the broken scale, $\hat\alpha$ still correlates strongly in **rank** with
   $\chi_{\text{event}}$ (Spearman $\rho=0.66$) — cross-validating the two independent diagnostics.
2. A diagnostic restricted to **only the matched events** (n=84,776, 11.5% of the total,
   avoiding the zero-floor entirely) shows a Laplace-scale correlation of **0.170**
   (95% CI 0.163–0.178, excludes 0 with a large margin).

**Conclusion — SIGNAL PARTIAL:** there IS a positive, statistically robust magnitude
coupling conditional on a match occurring, but it is **weaker** than the daily-block signal
from Gate G1. **Recommendation:** reformulate F5 as a **two-stage model** — (i) coincidence
probability (already characterized by C1b's coincidence-rate-vs-null), (ii) magnitude
conditional on matching (fit Heffernan–Tawn on the matched-only subset, not the
zero-substitution used here) — rather than abandoning F5, or naively proceeding with the
full flexible-$\beta$ fit on the flawed zero-substituted design. F6 (anisotropy/propagation
speed) remains weaker-than-expected but the non-null 0.17 signal keeps it open pending the
two-stage F5 refit.

**Output:**
- `results/gates/f5_pilot_results.parquet` — 6,132 directional pairs with $\hat\alpha$, CI, chi comparisons
- `results/gates/f5_pilot_decision.md` — written decision with date
- `results/figures/f5_pilot_alpha_vs_chi.png`

In [ ]:
run("F5_heffernan_tawn_pilot.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f5_pilot_alpha_vs_chi.png")))

---
### F5 (two-stage) + Gate G3 — reformulated conditional dependence model

Following the artifact found in the single-stage pilot (zero-substitution for unmatched
events), F5 is reformulated as a **two-stage (hurdle) model** — the standard approach for
sparse point-process data with excess zeros:

**Stage 1 — coincidence probability.** A first attempt at `logit(matched) ~ log(dist_km) +
X_i` found `dist_km` and the coincidence-window size (`dt_window_min`, set in C1b as
`dist_km / cloud_speed`) correlated at **0.99999** — perfectly collinear by design within
the 5 km cutoff. A raw distance coefficient therefore mechanically confounds genuine
physical coupling with "farther pairs get a longer search window". **Fix:** fit with an
**offset** for `logit(p_null)` (the independence-Poisson probability, itself a function
of the window) — the fitted coefficients now test only the **excess** over chance. The
null is reconstructed **directionally** ($p_{\text{null}} = 1 - e^{-\lambda_j\,2\Delta t}$
for the $i\to j$ direction, using the *partner* station's ramp rate $\lambda_j$),
replicating C1b's exact formula rather than its pooled per-pair average:

$$\text{logit}(P(\text{matched})) = \text{logit}(p_{\text{null}}) + \gamma_0 + \gamma_{\text{dist}}\log(\text{dist}_{km}) + \gamma_X X_i^{\text{Laplace}}$$

**Stage 2 — magnitude given coincidence.** Heffernan–Tawn ($Y_j = \alpha(\text{dist})\cdot X_i + X_i^\beta Z$)
fit **only on the 84,776 events that actually match** (not the zero-substituted set) —
this alone eliminates the pilot's artifact. $\alpha(\text{dist})$ tested in exponential vs.
power-law form via AIC — this directly answers F6's "exponential vs. power-law" question
through an independent channel from $\chi$. Parameters via MLE (Nelder–Mead), 95% CI via
**cluster bootstrap resampling directional pairs** (not individual events, to respect
within-pair correlation).

**Gate G3 criterion** (ROADMAP: "$\alpha$ coherent with G1's empirical $\chi$"): checked by
(a) same sign as $\hat\chi$ (non-negative by construction) and (b) Spearman correlation
between a naive per-pair $\hat\alpha$ (matched-only diagnostic, $n\geq15$) and G1's daily-block
$\hat\chi$ on the same pairs, against the coherence threshold declared in `config.py` (0.30).

> **Reading $\alpha$ correctly:** $\alpha_0$ is the **intercept at distance→0**. At the
> *median* matched-pair distance (~3.15 km), $\alpha(\text{dist})\approx0.24$ — the typical
> coupling. $\alpha$ is a regression **slope** ($Y_j \sim \alpha X_i$ on Laplace margins),
> **not** a correlation nor a $\chi$; the headline $\alpha_0=0.564$ should not be read as
> strong tail dependence (the pilot's raw matched-only correlation was 0.17).

**Output:**
- `results/gates/f5_stage1_coincidence_model.md` — Stage 1 logistic-with-offset coefficients
- `results/gates/f5_stage2_params.parquet` — Stage 2 winning-model parameters + bootstrap CI
- `results/gates/f5_stage2_pairwise_diagnostic.parquet` — naive per-pair $\hat\alpha$ (diagnostic only, $n\geq15$)
- `results/gates/gate3_decision.md`
- `results/figures/f5_stage1_calibration.png`, `results/figures/f5_stage2_alpha_vs_distance.png`

In [ ]:
run("F5_two_stage.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f5_stage1_calibration.png")))
display(Image(filename=str(ROOT / "results/figures/f5_stage2_alpha_vs_distance.png")))

---
### F6 — Spatial structure: anisotropy vs. real wind direction

F6's first question ("$\hat\chi$ vs. distance: exponential or power-law?") was already
answered by an independent channel in F5 Stage 2 ($\alpha(\text{dist})$ — exponential
won by AIC). This section covers the remaining question: **anisotropy** — is the
coincidence-excess signal (Gate G1/F5) stronger when the real ambient wind blows from the
conditioning station toward its neighbor, than when perpendicular/opposite?

**Design adaptation:** the ROADMAP's original plan (estimate propagation velocity/direction
from the time-lag between paired events) was invalidated by C1b's finding that event-to-event
magnitude coupling is weak — the Gate G1 dependence is mostly **shared regime**, not local
propagation. Instead, this reuses F5 Stage 1's design exactly (excess-coincidence logistic
regression, offset by the directional Poisson null) and adds one covariate:

$$\text{alignment}_{ij} = \cos(\text{travel\_dir}_{\text{wind}} - \text{bearing}_{i\to j})$$

Run automatically for **all 4 available wind sources** (KNMI 10m surface + CERRA
100/200/500m, once the CERRA download finished) sampled at the exact instant of each
conditioning extreme event, as an explicit robustness check across heights — no height
assumed correct a priori.

**Result — NOT stable across height:**

| Wind height | $\hat\gamma_{\text{align}}$ | 95% CI (bootstrap) | p | relative Δ | Meaningful? |
|---|---|---|---|---|---|
| KNMI 10m | -0.0189 | (-0.030, -0.006) | 0.0004 | -3.3% | No |
| CERRA 100m | -0.0267 | (-0.039, -0.016) | <0.0001 | -4.7% | No |
| CERRA 200m | -0.0331 | (-0.046, -0.023) | <0.0001 | -5.8% | **Yes** |
| CERRA 500m | -0.0458 | (-0.058, -0.035) | <0.0001 | -8.1% | **Yes** |

At 200m and 500m the effect crosses the 5% practical-relevance threshold — but with a
**wrong sign** relative to simple advection: coincidence is *higher* when wind blows
*against* the pair than *toward* it, and this gets stronger (more negative) with height.
This is not the pattern simple cloud advection predicts. Registered as "anisotropy
detected, sign inverted — investigate before interpreting physically", not as confirmed
anisotropy. Most likely explanation: wind direction aloft is less noisy/more
seasonally-structured than surface wind, so `alignment` at height may be picking up
residual seasonal/synoptic confounding with the shared regime already driving Gate G1,
rather than a genuine spatial transport mechanism.

**Decision:** ambiguous — see F6b below (the stronger, more direct timing test) for the
result that actually resolves this. Do not report the F6 sign-inverted effect alone as
"anisotropy evidence" in the paper.

**Output (per height, suffix `_10m`/`_100m`/`_200m`/`_500m`, plus a comparison):**
- `results/gates/f6_anisotropy_model_<height>.md`, `f6_anisotropy_bootstrap_<height>.parquet`, `f6_decision_<height>.md`
- `results/figures/f6_anisotropy_bins_<height>.png`
- `results/gates/f6_anisotropy_comparison.md`, `results/figures/f6_anisotropy_comparison.png` — cross-height summary

In [ ]:
run("F6_anisotropy.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f6_anisotropy_comparison.png")))
display(Image(filename=str(ROOT / "results/figures/f6_anisotropy_bins_500m.png")))

---
### F6b — Advection timing test: observed vs. expected lag (dist/wind speed)

The F6 test above only used wind **direction** as a covariate on whether a match occurs.
This is a stronger, more direct test of the physical advection mechanism, combining
distance + REAL wind speed + time: if a cloud genuinely travels from station $i$ to $j$
carried by the wind, the OBSERVED lag between the conditioning extreme event and its
matched partner event (`dt_min`, from `C1b_event_pairing.py`) should approach the
EXPECTED lag under pure advection: $\text{dist}_{ij}/v_{\text{along}}$, where
$v_{\text{along}}$ is the real KNMI wind speed projected onto the pair's geodesic bearing.

**Pre-registered expectation (stated by the user before first running with surface wind):**
no meaningful signal expected with 10m wind — it does not carry the clouds casting shadows
(those move at cloud-base height, hundreds of meters up). Once CERRA finished downloading,
this script was re-run automatically across **all 4 wind heights** (10/100/200/500m) as
the test that actually decides the advection hypothesis.

**Result — NULL at ALL 4 heights, this is the test that decides:**

| Wind height | Spearman ρ | 95% CI (bootstrap) | p | Meaningful? |
|---|---|---|---|---|
| KNMI 10m | -0.0043 | (-0.018, 0.008) | 0.47 | No |
| CERRA 100m | -0.0119 | (-0.022, -0.002) | 0.03 | No |
| CERRA 200m | -0.0129 | (-0.024, -0.001) | 0.01 | No |
| CERRA 500m | -0.0459 | (-0.058, -0.034) | <0.0001 | No |

Even at 500m — closest to cloud-base height, and where F6 above showed its strongest
(sign-inverted) alignment effect — the TIMING test, which is the most direct and physically
demanding (requires getting both the *order* and the *magnitude* of the delay right), stays
far below the |ρ|>0.10 practical-relevance threshold. The trend is monotonically more
negative with height (same direction as the F6 finding) but never approaches being
decisive. OLS slopes are all near-zero (max $|{\text{slope}}|=0.0085$ at 500m, R²≤0.003)
— nowhere close to slope=1 that perfect advection would produce.

**Robustness check (up/down ramp direction, kept from the original 10m-only run):** 96.0%
of ALL matched events already share the same direction (down-down or up-up) without this
being imposed by the matching — restricting to "down" only (the most direct cloud-shadow
signal) still gives a null correlation at every height, ruling out direction-mixing as an
explanation for the absence of signal.

**Consolidated conclusion (F6+F6b, 4 heights tested):** no wind height — surface or
CERRA 100/200/500m — produces a temporally decisive advection signal. The strongest,
most direct test (this one, timing) is null everywhere. F6's sign-inverted effect at
200/500m is registered as an intriguing but ambiguous result, not evidence of anisotropy
for the paper (it is physically backwards, and the more direct timing test does not
corroborate any transport mechanism at the same heights). The Gate G1 tail dependence
remains best explained as **shared regional weather regime**, not individually-trackable
cloud advection — this strengthens, rather than weakens, the original decision to model
the dependence via copulas/conditional extremes (Heffernan-Tawn) instead of an explicit
transport mechanism.

**Output (per height, plus comparison):**
- `results/gates/f6b_timing_model_<height>.md`, `f6b_timing_bootstrap_<height>.parquet`, `f6b_decision_<height>.md`
- `results/figures/f6b_timing_scatter_<height>.png`
- `results/gates/f6b_timing_comparison.md`, `results/figures/f6b_timing_comparison.png` — cross-height summary

In [ ]:
run("F6b_wind_timing.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f6b_timing_comparison.png")))
display(Image(filename=str(ROOT / "results/figures/f6b_timing_scatter_500m.png")))

---
### S4 — Advection timing re-test with native 10-minute KNMI wind resolution (Risco C)

F6b above uses HOURLY surface wind (KNMI De Bilt), coarser than the phenomenon's own scale
(median ramp duration 14min, median inter-station lag also ~14min) — raising the concern
that the null advection result could be an artifact of coarse wind timing rather than a
genuine absence of physical signal. `B7d_wind_knmi_10min.py` (one-time download, not run in
this notebook — ~4h against KNMI's public anonymous-key 10-minute observations API) fetched
the KNMI 10-minute-native wind at the 6,571 unique rounded timestamps actually needed by the
matched events (6,570/6,571 = 100.0% coverage). This cell re-runs F6b's exact same central
analysis (same filters, same bootstrap) but joins wind at the nearest 10-minute timestamp
(residual misalignment ≤5min, vs. up to ±30min under the hourly join).

**Result:** Spearman ρ moves from -0.0043 (hourly, CI crosses 0, not even statistically
detectable) to -0.0134 (10min-native, statistically detectable at n=28,240 but still an
order of magnitude below the |ρ|>0.10 practical-relevance threshold). **F6b's null result is
confirmed as robust to wind temporal resolution** — coarse hourly wind timing was NOT hiding
a real advection signal.

**Output:**
- `results/gates/s4_f6b_timing_model_10min_hires.md`, `s4_f6b_decision_10min_hires.md`
- `results/gates/s4_f6b_resolution_comparison.md` — hourly vs. 10min side-by-side table
- `results/figures/s4_f6b_timing_scatter_10min_hires.png`

In [ ]:
run("S4_f6b_wind_10min.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/s4_f6b_timing_scatter_10min_hires.png")))

---
### F6c — Census of wind/cloud data sources considered for F6/F6b

Not a new statistical analysis — a formal record consolidating every spatially-resolved
wind/cloud data source evaluated throughout the project for the anisotropy/advection test,
whether it was actually downloaded and tested or discarded beforehand. Motivation: for
F6b's null advection-timing result (across all 4 tested heights) to credibly support
modelling the dependence via copulas/Heffernan-Tawn rather than an explicit advection
mechanism, the search for adequate wind data needs to be shown as reasonably exhaustive,
not stopped at the first available source.

**Result: 8 sources identified, 4 actually tested** (KNMI De Bilt 10m + CERRA
100/200/500m). Of the 4 discarded before testing, the reasons are **heterogeneous** — not
all due to insufficient granularity:
- **2 discarded for insufficient spatial granularity**: ERA5 (~31km — the whole Utrecht
  network fits in ~1.4×1.6 grid cells) and EUMETSAT Atmospheric Motion Vectors (~72×72km
  per vector — coarser than ERA5, already rejected).
- **2 discarded for other reasons**: KNMI-HARMONIE/KNW actually had *better* spatial
  resolution than CERRA (~2.5km vs 5.5km) but was dropped due to access friction (no
  simple anonymous key); Cabauw tower had *better* temporal resolution (10min vs CERRA's
  3h) but was dropped for insufficient net gain (lower height ceiling than CERRA, and
  likely non-independence since CERRA probably assimilates Cabauw as an input).
- Two additional cloud-cover (not wind) sources were also considered and dropped: KNMI's
  single-station cloud-oktas variable (redundant with the already-computed clearsky
  index, no directional component) and CLAAS-3/SEVIRI (would require building a custom
  cloud motion-vector algorithm — a scope increase equivalent to Paper 2, deliberately
  out of scope here).

**Output:**
- `results/gates/f6_wind_sources_census.md` — full citable table for the paper's Data and Methods section

In [ ]:
run("F6c_wind_sources_census.py")

---
### S7 — Seasonal stratification of the advection timing test (Aprimoramento G)

Checks whether F6b's aggregate null result conceals seasonal heterogeneity.
Pre-specified primary hypothesis: **JJA (summer) vs. rest of year** — convective clouds in summer
may yield a detectable advection timing signal that is diluted in the annual aggregate.
Exploratory: 4 seasons (DJF / MAM / JJA / SON), interpreted with Bonferroni correction.

Runs for all 4 wind heights (KNMI 10m surface + CERRA 100/200/500m) — 5 seasonal groups × 4
heights = 20 combinations.

**Result:** No season × height combination reached |ρ| > 0.10 (practical-significance threshold).
Notably, JJA shows **negative ρ at all heights** (KNMI: −0.052; CERRA 100m: −0.061; 200m: −0.064;
500m: −0.081), directly contradicting the summer-convection advection hypothesis. The null is not
concealing a seasonal signal — it is robust across all seasons.

See `results/gates/s7_seasonal_comparison.md` and heatmap `results/figures/s7_seasonal_comparison.png`.

In [ ]:
run("S7_f6b_seasonal.py")

---
### D2 — Informative censoring check (9 low-coverage stations)

Closes an open item flagged back in B2: 9 stations have overall coverage < 70%
(ID051, ID115, ID004, ID037, ID041, ID049, ID078, ID063, ID046). Does this missing data
concentrate on physically interesting (high tail-activity) days — which would bias
G1/G2/G3 by under-representing these stations' true extreme behaviour (truncation bias)?

**Method:** build a daily network-activity index (count of extreme ramps, |Δk| ≥ P95)
from 20 near-complete anchor stations; correlate each target station's daily coverage
against this index (Spearman, Benjamini–Hochberg FDR across the 9 tests). A robustness
check excludes long (≥20-day) outage blocks to rule out a simple seasonal-coincidence
confound.

**Result:** 8/9 stations show a statistically significant, FDR-robust NEGATIVE
correlation (ρ between -0.07 and -0.32) — these stations genuinely lose more data on
days when the network is more active, confirming informative censoring (not just a
seasonal artifact, since the pattern survives excluding long outage blocks).

**Output:**
- `results/gates/informative_censoring_results.parquet`
- `results/gates/informative_censoring_decision.md`
- `results/figures/d2_censoring_vs_activity.png`

In [ ]:
run("D2_informative_censoring.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/d2_censoring_vs_activity.png")))

---
### D2b — Sensitivity of G1/G2/G3 to excluding the 9 flagged stations

The censoring found in D2 is real, but is it *consequential*? Rather than re-fitting
each gate from scratch, this filters the already-computed gate outputs (G1's
`gate1_results`, G2's `gpd_marginal_params`, G3's per-pair diagnostic) to exclude the 9
stations and recomputes each gate's headline decision metric.

**Result — all three gates are stable:**
- G1: 52.1% → 52.5% of close pairs significant (excluding the 9 stations) — still far
  above the 30% GO threshold.
- G2: 98.0% → 98.2% pass rate — materially unchanged.
- G3: median per-pair α̂ 0.472 → 0.473 — no significant difference (Mann–Whitney
  p=0.43) between pairs involving vs. not involving the 9 stations.

**Conclusion:** the informative censoring is statistically real but practically
inconsequential for the three approved gates. The 9 stations are retained in the main
analysis with this limitation documented, rather than excluded.

**Output:** `results/gates/censoring_sensitivity_results.md`

In [ ]:
run("D2b_censoring_sensitivity.py")

---
## F7a — Aggregate network ramp series ("virtual power plant")

Treats the network as a single capacity-weighted virtual plant: k_agg(t) = Σ w_i·k_i(t),
w_i = capacity_i / Σcapacity. Reuses `clearsky_index.parquet` (B4) directly — cross-station
clearsky variation was already shown negligible in B4 (all 175 stations within ~10km of
Utrecht) — and the exact same SDT detector (B5, unchanged ε=0.02/Δ=0.10/min-duration=10min)
is applied to this single series. This avoids having to reconcile per-station ramp events
(which start/end at different times) into an ad hoc network-level event definition; the
network is treated as "just another station" for detection purposes.

**Result:** 4,630 train / 1,672 test aggregate ramp events (down: 2,365 train). Crude
aggregate/individual median-magnitude ratio ≈ 0.94 (down) / 0.95 (up) — much closer to 1
than one would expect under meaningful diversification, an early descriptive signal
consistent with the "shared regime" finding from C1b/F6b (formal test with a proper
independence counterfactual is done in F8 below).

In [ ]:
run("F7a_aggregate_series.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f7a_aggregate_series_example.png")))

---
## F7 — Gate G4: Return levels + backtesting

Fits the exact same adaptive-threshold-selection + Ferro-Segers-declustering POT/GPD
methodology as Gate G2 (`C2_gate2.py`) to the single aggregate series (down direction —
generation drops are the operationally relevant "need reserve" event). Confidence
intervals use a **moving block bootstrap over days** (reuses `estimate_block_length` /
`make_block_index_arrays` from Gate G1's `C1_gate1.py`, block_length=10 days, B=150), never
i.i.d. resampling, since ramp occurrence is temporally clustered.

**Fit (train 2014-2016, 3.00 years):** u=0.2150, ξ̂=−0.1951, σ̂=0.0735, 211 declustered
exceedances (θ̂=0.595).

| T (years) | z_T | 95% CI (block bootstrap) |
|---|---|---|
| 1/12 | 0.325 | (0.312, 0.340) |
| 1/4 | 0.376 | (0.360, 0.392) |
| 1/2 | 0.404 | (0.385, 0.421) |
| 1 | 0.427 | (0.405, 0.450) |
| 2 | 0.448 | (0.420, 0.479) |
| 5 | 0.472 | (0.435, 0.514) |

**Backtest (Gate G4):** the train-fitted return levels imply an expected Poisson
exceedance rate; comparing against the 2017 holdout year (never seen in the fit) via a
Poisson prediction interval (the POT/GPD analogue of the Kupiec VaR-coverage test) — **4/4
tested horizons (1/12, 1/6, 1/4, 1/2 year) covered (100%)**. **Gate G4: APPROVED.**

*Note on the complementary sanity check in the figure below:* the observed 2017 annual max
(0.513) sits ABOVE the 1-year return level's 95% CI (0.405–0.450). This is expected, not a
model failure — a POT/GPD return level is defined by the exceedance RATE (1 exceedance
expected every T years), so for a Poisson-GPD process P(annual max > z₁) ≈ 1−1/e ≈ 63% by
construction (Coles 2001, ch. 4). The formal coverage test is the Poisson exceedance-count
backtest above, not this point-in-time comparison.

In [ ]:
run("F7_return_levels.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f7_return_level_curve.png")))
display(Image(filename=str(ROOT / "results/figures/f7_backtest_coverage.png")))

---
## F8 — RQ3: Portfolio effect under independence vs. real dependence

Three scenarios, all using the exact same POT/GPD + block-bootstrap methodology as F7/Gate
G4, so the comparison is apples-to-apples:

1. **REAL** — the observed aggregate series (F7a/F7, Gate G4 approved).
2. **INDEPENDENCE** — each station gets an INDEPENDENT circular day-block shift (150
   realizations, 143 converged; block_length=10 days, same as F7). This exactly preserves
   every station's individual margin/GPD but destroys spatial synchrony — it is the
   "portfolio diversification" effect assumed by naive reserve-sizing practice, with no
   parametric dependence assumption at all (a pure empirical counterfactual).
3. **MODEL-IMPLIED** — starts from the real training series; for each of the 20,614 real
   historical extreme events, replaces the response of neighbors <5.0km away with a draw
   from the two-stage model already approved at Gate G3 (`F5_two_stage.py`: Stage 1
   offset-logistic decides coincidence, Stage 2 Heffernan-Tawn draws the magnitude). 150/150
   realizations converged. This is a validation of the conditional-extremes model, not the
   central RQ3 result (that's REAL vs. INDEPENDENCE).

**Tab. 5 — Return levels by scenario and RQ3 ratio**

| T (years) | z_real | z_independence | z_model-implied | Ratio (real/indep) | 95% CI | Δ model vs. real |
|---|---|---|---|---|---|---|
| 1/12 | 0.325 | 0.126 | 0.303 | 2.572 | (2.409, 2.743) | −6.8% |
| 1/4 | 0.376 | 0.148 | 0.355 | 2.536 | (2.299, 2.762) | −5.6% |
| 1/2 | 0.404 | 0.164 | 0.384 | 2.470 | (2.192, 2.739) | −5.0% |
| 1 | 0.427 | 0.178 | 0.410 | 2.388 | (2.033, 2.698) | −4.1% |
| 2 | 0.448 | 0.195 | 0.433 | 2.293 | (1.853, 2.670) | −3.4% |
| 5 | 0.472 | 0.219 | 0.460 | 2.168 | (1.611, 2.691) | −2.5% |

**RQ3 central result (T=1 year): real/independence ratio = 2.388, 95% CI [2.033, 2.698]**
— the CI excludes 1, so real spatial dependence requires statistically significant
additional reserve capacity relative to the naive independence heuristic. Framed as the
headline number: **assuming independence underestimates the true required reserve by 139%
(95% CI [103%, 170%]) at the 1-year horizon.**

**Model validation:** the model-implied return levels track the real ones within a median
4.5% deviation across all 6 horizons — the two-stage Heffernan-Tawn model (Gate G3)
reproduces the real portfolio tail reasonably well, evidence that the conditional-extremes
apparatus has genuine predictive utility for aggregate risk, not just a pairwise post-hoc
fit.

**RQ3: CONFIRMED.** This closes the pilot's three research questions (RQ1: Gate G1 spatial
tail dependence — CONFIRMED; RQ2: Gate G3 distance-decay coupling α(dist) — CONFIRMED;
RQ3: portfolio effect — CONFIRMED, independence underestimates reserve need).

In [ ]:
run("F8_portfolio_effect.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/f8_reserve_comparison.png")))

---
## F8b — RQ3b: Benchmark against current industry practice (Gaussian copula / Pearson correlation)

F8 compares REAL vs. INDEPENDENCE (the worst case). This does **not** answer the question a
power-systems reviewer asks first: "compared to what the industry/literature already does
today?" F8b closes that gap with two additional scenarios:

1. **GAUSSIAN COPULA (conditional)** — mirrors F8's model-implied scenario EXACTLY (same
   Stage-1 coincidence GLM, same conditioning events, 150 realizations) but replaces the
   Heffernan-Tawn Stage-2 draw with a conditional Gaussian copula, ρ(dist) fit via Pearson
   correlation on the same Laplace margins used for α(dist) (form: power, ρ₀=0.248).
2. **CLOSED-FORM FALLBACK** — the classical weighted-sum portfolio variance formula
   (Var(k_agg)=Σwᵢ²σᵢ²+ΣΣwᵢwⱼρᵢⱼσᵢσⱼ) using the REAL Pearson correlation matrix of the raw
   k_i(t) signal (no coincidence model at all) — the crudest, most-cited industry practice.

**Critical finding:** z_copula (0.410 at T=1y) and z_model-implied (0.410, F8's Heffernan-Tawn
scenario) are almost identical (0.1% difference) — because both scenarios share the identical
Stage-1 coincidence mechanism and only differ in Stage-2 (magnitude-given-coincidence: Gaussian
linear vs. Heffernan-Tawn nonlinear). **Once coincidence (shared-regime) is correctly modeled,
the magnitude dependence structure matters comparatively little** — reinforcing C1b/F5's
central finding that dependence is predominantly shared-regime, not event-to-event magnitude
coupling. The benchmark that actually exposes the fragility of a naive, coincidence-free
Gaussian/Pearson practice is the **closed-form fallback**: ratio real/closed-form = **1.311**
at T=1y — much closer to the independence worst-case (2.388) than to parity. Report both
numbers in the paper, with the explicit distinction between "copula with coincidence modeled"
(does not underestimate) and "pure linear practice without a coincidence model" (underestimates
by ~31%).


In [ ]:
run("F8b_industry_benchmark.py")


In [ ]:
display(Image(filename=str(ROOT / "results/figures/f8b_reserve_comparison.png")))
display(Image(filename=str(ROOT / "results/figures/f8b_rho_vs_alpha.png")))


---
## S8 — Is F8b's Gaussian-copula "tie" a shared-architecture artifact? (Aprimoramento I)

F8b's Gaussian-copula scenario mirrors F8's model-implied scenario EXACTLY (same Stage-1
coincidence GLM) — raising the question of whether the near-tie is a real data property or
an artifact of reusing the same coincidence machinery for both scenarios. S8 answers this
with a standard EVT diagnostic (Coles, Heffernan & Tawn, 1999) that is **deliberately
independent** of the F5/F8/F8b architecture: it uses the raw Gate G1 daily-max data (3,070
pairs < 5km) directly, calibrates a Gaussian copula per pair via Pearson correlation on
normal scores, and compares the empirical tail-dependence coefficient χ̂(u) against the
Gaussian-implied χ(u) at the *same* correlation, via the exact bivariate-normal CDF (not
the asymptotic χ̄=ρ shortcut).

**Result: no artifact — the tie reflects genuine data structure.** χ̂ empirical is, if
anything, *below* χ_gaussian at both tested quantiles (u=0.90: median diff -0.019, 95% CI
[-0.063, 0.021], not significant; u=0.95: median diff -0.030, 95% CI [-0.065, -0.004],
significantly negative). There is **no evidence of heavier-than-Gaussian tail dependence**
at this granularity — a 4th independent diagnostic (alongside F6/F6b's null advection
result and C1b's shared-regime finding) converging on the same conclusion: the paper's
advantage over naive market practice comes from **explicitly modeling event coincidence**
(Stage 1 / shared regime), not from the magnitude-copula functional form. See
`PRE_PAPER.md` Section 0.1.1 for the corresponding framing guidance.

**Output:**
- `results/gates/s8_chi_vs_gaussian.parquet`, `s8_chi_vs_gaussian_decision.md`
- `results/figures/s8_chi_vs_gaussian.png`

In [ ]:
run("S8_chi_vs_gaussian_copula.py")


In [ ]:
display(Image(filename=str(ROOT / "results/figures/s8_chi_vs_gaussian.png")))


---
## S9 — Definitive advection matrix: all heights × all seasons (B7c data)

B7c (CERRA pressure levels, 72 months) is now complete. `S9_advection_full_matrix.py`
extends the advection timing test (F6b / S4 / S7) to the **full atmospheric column**:
12 wind sources (KNMI 10m + CERRA 100/200/500m + CERRA 950/900/850/800/700/600/500/400hPa)
× 5 seasonal epochs × 2 modes (with/without ±30min CERRA colocation filter) = **120 cells**.

**Central question:** does any height, in any season, show a practically relevant
advection timing signal (|ρ| > 0.10) between pairs of PV stations?

**Result: NO for all pre-specified groups.** Mode A (no filter): |ρ|_max = 0.083 across all
12 heights × JJA and REST seasons. Mode B (±30min colocation): 2 exploratory cells (DJF ×
CERRA 500hPa/400hPa, ρ ≈ 0.10-0.11) do not survive Bonferroni correction for 120 tests and
are consistent with synoptic winter-wind confounding already documented in F6_anisotropy.py.

**Physical conclusion:** individual clouds do NOT generate a measurable advection timing
signal at any layer of the atmospheric column (surface to ~7.2km) in any pre-specified
season. Spatial dependence (Gate G1, χ > 0) is driven by **shared meteorological regime**
(meso/macro-scale systems affecting multiple stations simultaneously), not physical cloud
transport — a 5th independent diagnostic (F6b/S4/S7/S8/S9) converging on the same mechanism.

**Outputs:**
- `results/gates/s9_full_matrix.parquet`, `s9_full_matrix_decision.md`
- `results/figures/s9_full_matrix_heatmap.png` — heatmap ρ (season × height, both modes)
- `results/figures/s9_full_matrix_profile.png` — vertical profile of ρ by season

In [ ]:
run("S9_advection_full_matrix.py")

In [ ]:
display(Image(filename=str(ROOT / "results/figures/s9_full_matrix_heatmap.png")))
display(Image(filename=str(ROOT / "results/figures/s9_full_matrix_profile.png")))

---
## F6d — Vetor de deslocamento de nuvem implícito pela rede (5º diagnóstico de advecção)

> **Script:** `F6d_network_wind_vector.py`  
> **Método:** Bosch & Kleissl (2013) — inversão OLS por dia sobre todos os pares casados <5km
> simultaneamente. Em vez de assumir uma altura de vento a priori e testar a correlação
> tempo/distância (como F6b), **inverte o problema**: deixa os dados revelar qual vetor de
> velocidade (módulo + direção) melhor explica todos os atrasos observados numa mesma janela,
> sem usar nenhuma fonte externa.
>
> **Fase 1 (sem vento externo):** 702 dias com ≥5 pares casados. Velocidade implícita mediana:
> **0,11 m/s**; **0 dias (0%)** com vetor não-degenerado ≥1 m/s; R² mediano do ajuste por
> janela = 0,012. Resultado mais decisivo que F6b: mesmo dando ao conjunto de dados a melhor
> oportunidade de mostrar advecção, nada emerge.
>
> **Fase 2 (12 alturas: KNMI 10m + CERRA 100/200/500m + 8 níveis de pressão 950–400 hPa):**
> inviabilizada pelo resultado da Fase 1 — com 0 vetores não-degenerados não há o que
> correlacionar contra o vento real. A ausência de janelas utilizáveis IS the result.
>
> **5º diagnóstico independente confirmando regime compartilhado** (sequência: F6 → F6b → S4
> → S7+S9 → S8 → F6d). Veja `results/gates/f6d_wind_comparison.md`.

In [ ]:
# F6d — Vetor implícito OLS (Fase 1 + Fase 2 completa)
# Fase 1: 702 dias, mediana=0.11m/s, 0 dias não-degenerados (>=1m/s), R2=0.012
# Fase 2: 0 janelas utilizáveis em 12 alturas → inviabilizada (o próprio resultado)
import subprocess
result = subprocess.run(['python', 'F6d_network_wind_vector.py'], capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib

fig = pathlib.Path('results/figures/f6d_network_vector_examples.png')
if fig.exists():
    display(Image(str(fig), width=700))

report = pathlib.Path('results/gates/f6d_wind_comparison.md')
if report.exists():
    display(Markdown(report.read_text()))

---
## S5/S6 — Risco D: genuine out-of-sample validation of F8's "model-implied" scenario

F8's model-implied scenario applies the two-stage model back onto the SAME training-period
conditioning events used to fit it — its reported 4.5% median deviation is an in-sample
fit-quality metric, not out-of-sample validation. **S5** rebuilds the event-pairing catalog
restricted to the 2017 holdout (extremity threshold learned purely on 2014-2016, neighbor
catalog restricted to 2017 — genuinely never seen by the model fit). **S6** applies the
already-fitted model (no retraining) to these 2017 events, patching the REAL 2017 aggregate
series, and checks two things against the real 2017 series:

(a) **Exceedance backtest** (same methodology as Gate G4): counts exceedances of the
train-fitted return level z_T in the model-implied-2017 series vs. the real 2017 series vs.
the Poisson predictive interval.
(b) **GPD return level fit directly on 2017** (years_span=1): median % deviation between the
model-implied-2017 and real-2017 return levels — the genuinely out-of-sample version of F8's
4.5% statistic.

**Result: 100% Poisson coverage (4/4 backtest horizons) — identical to the real 2017 series
(Gate G4).** Median % deviation = **6.5%** (comparable to F8's in-sample 4.5%). **RISCO D
RESOLVED: the two-stage model has genuine predictive utility for data it never saw during
fitting — not just a good in-sample fit.** Does not affect RQ3's headline ratio (2.39×, which
never uses the two-stage model).


In [ ]:
run("S5_event_pairing_test.py")


In [ ]:
run("S6_f8_oos_validation.py")


In [ ]:
display(Image(filename=str(ROOT / "results/figures/s6_oos_validation.png")))


---
## S10 — Network density sensitivity (station subsampling bootstrap)

A reviewer may ask whether Gate G1 (52.1% significant close pairs) and the Stage-2 decay length
($\hat{L} = 3.72$ km) depend on the **specific spatial density** of the 174-station Utrecht
network, rather than being stable properties of the region.

`S10_subsampling_bootstrap.py` addresses this by drawing **100 random subsamples** without
replacement for each $k \in \{50, 80, 100, 120, 150\}$ stations (pool: 174 systems with valid
coordinates). For every subsample, using **training data only (2014–2016)**:

1. **Gate G1** — daily-block $\hat{\chi}(u=0.95)$, block permutation ($N=199$), Benjamini–Hochberg
   FDR within the close-pair family ($d \leq 5$ km); record the fraction of significant pairs
   and the median $\hat{\chi}$ among significant pairs.
2. **Stage 2** — filter the existing matched-event catalogue to retained stations; refit
   $\alpha(d)=\alpha_0 e^{-d/L}$ by MLE with **$\beta=0.399$ fixed** at the production value.

**Decision criterion:** *stable* if the G1 fraction stays above the 30% gate threshold and
median $\hat{L}$ remains within the production 95% CI $[3.41, 4.02]$ km for $k \geq 80$.

**Result (2026-07-04): STABLE.** All 500 replicates completed successfully. Median G1 fraction
$\approx 51\%$ at every $k$; median $\hat{L}$ at $k=80$: 3.83 km; at $k=150$: 3.72 km.
Integrated in the paper as **Appendix A.6** (`app:subsampling`, Figure `fig:subsampling`) and
closes the network-density limitation in Discussion §5.4.

> **Runtime:** ~5 min for 100 replicates per $k$ (`python S10_subsampling_bootstrap.py`).
> Use `--reps 20` for a quick smoke test.

**Outputs:**
- `results/gates/s10_subsampling_bootstrap.parquet`
- `results/gates/s10_subsampling_decision.md`
- `results/figures/s10_subsampling_stability.pdf`
- `paper/figures/s10_subsampling_stability.pdf` (LaTeX copy)

In [ ]:
run("S10_subsampling_bootstrap.py")

In [ ]:
from IPython.display import Markdown, display

decision = ROOT / "results/gates/s10_subsampling_decision.md"
if decision.exists():
    display(Markdown(decision.read_text()))

display(Image(filename=str(ROOT / "results/figures/s10_subsampling_stability.pdf")))

---
## S11 — Statistical details for supplementary tables

Before generating LaTeX tables (`P_tab*`), this step consolidates **inference-ready
statistics** already computed by the pipeline into a single long-format CSV suitable
for Overleaf ingestion or manual table assembly.

`S11_statistical_details.py` reads precomputed artefacts (no new modelling beyond a
fast refit of Stage 1) and writes `results/gates/s11_statistical_details.csv` with
columns `metric, value`.

**Extracted quantities:**

1. **Stage 1 logistic regression** (same specification as `F5_two_stage.py`):
   logit(P(match)) = logit(p_null) + γ₀ + γ_d·log(dist) + γ_X·X_i(Laplace).
   For each coefficient: estimate, standard error, z-value, p-value, and 95% CI
   (Wald intervals from the statsmodels GLM fit).

2. **Gate G1 p-values:** among the 1,599 significant close pairs (d ≤ 5 km, BH-adjusted
   at u = 0.95), percentiles p10–p90 of the adjusted p-values.

3. **System counts:** total stations in the Utrecht dataset (175), stations with valid
   coordinates (174), and stations entering spatial analyses (174).

**Reference values (paper):** γ₀ = 3.68, γ_d = −0.651, γ_X = −0.139, pseudo-R² = 0.93.

> **Runtime:** ~5 s (`python S11_statistical_details.py`). Requires `C1_gate1.py`,
> `C1b_event_pairing.py`, and `F5_two_stage.py` outputs.

**Outputs:**
- `results/gates/s11_statistical_details.csv`

In [ ]:
run("S11_statistical_details.py")

In [ ]:
import pandas as pd

s11_csv = ROOT / "results/gates/s11_statistical_details.csv"
if s11_csv.exists():
    display(pd.read_csv(s11_csv))
else:
    print(f"Not found: {s11_csv}")

---
## Section 10 — Paper figures & tables (P_* scripts)

All publication-quality figures and tables for the paper are generated here.
Each script reads from pre-computed parquets in `results/gates/` and writes to:
- `paper/figures/` — vector PDF figures (600 dpi embedded rasters where needed)
- `paper/tables/` — LaTeX table fragments

Run **S10** (Section above) before this section if the subsampling robustness figure
(`s10_subsampling_stability.pdf`) is not yet present.

Run **S11** (Section above) if `s11_statistical_details.csv` is not yet present —
supplementary inference statistics for Stage 1, Gate G1, and system counts.

Reference: `paper/RESULTS_STORYBOARD.md`


In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path("..")  # workspace root

PAPER_SCRIPTS = [
    # ── Main text figures ──────────────────────────────────────────────────
    "P_fig41_chi_decay.py",          # Fig 4.1 — chi vs. distance (Gate G1)
    "P_fig42_regime_asymmetry.py",   # Fig 4.2 — shared regime vs. event coupling
    "P_fig43_advection_matrix.py",   # Fig 4.3 — advection null matrix (S9)
    "P_fig44_two_stage_model.py",    # Fig 4.4 — Stage 1 + Stage 2 model
    "P_fig45_return_levels.py",      # Fig 4.5 — return levels + Gate G4
    "P_fig46_reserve_comparison.py", # Fig 4.6 — 5-scenario reserve (CENTRAL)
    "P_fig47_validation_composite.py",# Fig 4.7 — OOS validation + chi vs. Gaussian
    # ── Main text tables ──────────────────────────────────────────────────
    "P_tab41_gate1.py",              # Tab 4.1 — Gate G1 summary
    "P_tab42_model_params.py",       # Tab 4.2 — Stage 1+2 parameters
    "P_tab43_return_levels.py",      # Tab 4.3 — return levels by scenario
    # ── Appendix A ────────────────────────────────────────────────────────
    "P_figA1_sdt_sensitivity.py",    # Fig A.1 — sharp/gradual contrast vs. threshold
    "P_figA2_block_sensitivity.py",  # Fig A.2 — ratio vs. block length (S1)
    "P_figA3_gpd_quality.py",        # Fig A.3 — GPD shape distribution
    "P_figA4_censoring.py",          # Fig A.4 — informative censoring (D2)
    "P_tabA1_data_quality.py",       # Tab A.1 — Gate G0 + D2 summary
    "P_tabA2_gpd_params.py",         # Tab A.2 — GPD parameters summary
    # ── Appendix B ────────────────────────────────────────────────────────
    "P_figB1_advection_timing_grid.py", # Fig B.1 — timing grid 4 heights × 2 seasons
    "P_figB2_spatial_coherence.py",     # Fig B.2 — sharp vs. gradual coherence (F2)
]

ok, failed = [], []
for script in PAPER_SCRIPTS:
    path = ROOT / script
    result = subprocess.run(
        [sys.executable, str(path)],
        capture_output=True, text=True,
        cwd=str(ROOT)
    )
    if result.returncode == 0:
        ok.append(script)
        print(f"  ✓ {script}")
    else:
        failed.append(script)
        print(f"  ✗ {script}")
        print(result.stderr[-300:] if result.stderr else result.stdout[-300:])

print(f"\n{'='*50}")
print(f"Paper artifacts: {len(ok)}/{len(PAPER_SCRIPTS)} OK")
if failed:
    print(f"FAILED: {failed}")
else:
    print("All scripts passed.")


---
## Outputs summary

| Location | File | Description |
|---|---|---|
| `data/interim/` | `coords.parquet` | 174 station centroids (lat, lon, capacity) |
| `data/interim/` | `power_qc.parquet` | Raw power after QC and operational clipping |
| `data/interim/` | `power_norm.parquet` | Normalized power $p^*_i(t)$ |
| `data/interim/` | `clearsky_index.parquet` | Clearsky index $k_i(t)$ + split label |
| `data/interim/` | `ramps.parquet` | All detected ramp events |
| `data/interim/` | `ramps_split.parquet` | Ramps with train/test label |
| `data/interim/` | `wind_joined.parquet` | Ramps + wind covariates: real 10m (KNMI De Bilt, B7) + optional 100/200/500m (CERRA, B7b) |
| `data/raw/wind/` | `knmi_debilt_wind.csv`, `cerra_by_year/cerra_<year>_<mm>-<mm>.nc` (24 quarterly files) | Cached raw wind downloads (B7, B7b) |
| `data/interim/` | `sensitivity_grid.csv` | $(\varepsilon, \Delta)$ SDT sensitivity results (B6) |
| `results/figures/` | `B5_visual_check.png` / `B5_visual_check_zoom.png` | Visual validation of detected ramps (B5b) |
| `results/gates/` | `f2_spatial_coherence.csv` | Distance vs. lag pairs, sharp vs. gradual ramps (F2) |
| `results/figures/` | `F2_lag_vs_distance.png` | Spatial coherence comparison figure (F2) |
| `data/interim/` | `qc_report.csv` | Per-station censorship fractions |
| `data/interim/` | `split_report.csv` | Per-station event counts by split |
| `results/gates/` | `gate0_results.parquet` | Pairwise RU from Gate G0 MC |
| `results/gates/` | `gate0_decision.md` | Gate G0 written decision (**APPROVED WITH CAVEAT**) |
| `results/figures/` | `gate0_ru_histogram.png` | RU distribution across all pairs |
| `results/figures/` | `gate0_ru_vs_dist.png` | RU vs nominal distance scatter |
| `data/processed/` | `uniform_margins.parquet` | Rank-based uniform margins, daily block maxima (Gate G1 E.1) |
| `results/gates/` | `chi_estimates_raw.parquet` | χ̂/χ̄ for all 15,051 pairs, u∈{0.90,0.95} (Gate G1 E.3) |
| `results/gates/` | `gate1_results.parquet` | 3,070 close pairs: χ̂, χ̄, bootstrap CI, p-value, FDR flag |
| `results/gates/` | `gate1_decision.md` | Gate G1 written decision (**G1 APPROVED**, 52.1% significant) |
| `results/figures/` | `gate1_chi_vs_distance.png` | χ̂ vs. distance, all pairs + significant highlighted |
| `results/figures/` | `gate1_ci_examples.png` | Bootstrap CI examples for random close pairs |
| `data/processed/` | `aligned_pairs.parquet` | Event-level matched extreme events per close pair (Gate G1 refinement, C1b) |
| `results/gates/` | `event_pairing_summary.parquet` | Per-pair χ_event, coincidence rate vs. independence null (C1b) |
| `results/gates/` | `gate1_event_refinement.md` | Event-level refinement finding (regime- vs. event-level dependence) |
| `results/figures/` | `gate1b_chi_event_vs_daily.png` | χ_event (nearest / any-in-window) vs. daily-block χ̂ |
| `results/figures/` | `gate1b_coincidence_vs_null.png` | Observed vs. null coincidence rate by distance |
| `results/gates/` | `gpd_marginal_params.parquet` | GPD params (ξ, β₀, β₁, β₂) per station × direction (Gate G2), incl. no-covariate cross-check columns |
| `results/gates/` | `gate2_decision.md` | Gate G2 written decision (**G2 APPROVED**, 98.0% of series pass after cross-check) |
| `results/figures/` | `gate2_threshold_diagnostics.png` | MRL + parameter stability plots, 5 sample stations |
| `results/figures/` | `gate2_qq_plots.png` | GPD QQ-plots, same 5 sample stations |
| `results/figures/` | `gate2_extremal_index.png` | Distribution of extremal index θ across series |
| `results/gates/` | `f5_pilot_results.parquet` | α̂ (β=0 pilot) per directional pair, vs. χ_event/χ_daily |
| `results/gates/` | `f5_pilot_decision.md` | F5 feasibility pilot decision (**SIGNAL PARTIAL**) |
| `results/figures/` | `f5_pilot_alpha_vs_chi.png` | α̂ vs. χ_daily and χ_event scatter plots |
| `results/gates/` | `f5_stage1_coincidence_model.md` | Stage 1 logistic-with-offset coefficients (excess coincidence) |
| `results/gates/` | `f5_stage2_params.parquet` | Stage 2 winning model (exp vs. power-law by AIC) params + bootstrap CI |
| `results/gates/` | `f5_stage2_pairwise_diagnostic.parquet` | Naive per-pair α̂ (matched-only, diagnostic, n≥15) |
| `results/gates/` | `gate3_decision.md` | Gate G3 written decision (**G3 APPROVED**, α coherent with G1's χ) |
| `results/figures/` | `f5_stage1_calibration.png` | Stage 1 observed vs. predicted vs. null coincidence, by distance decile |
| `results/figures/` | `f5_stage2_alpha_vs_distance.png` | α(dist) fit vs. per-pair diagnostic; comparison with G1's χ̂ decay |
| `results/gates/` | `informative_censoring_results.parquet` | Per-station coverage-vs-activity correlation (raw + robust), 9 low-coverage stations |
| `results/gates/` | `informative_censoring_decision.md` | D2 written decision (**INFORMATIVE, confirmed robust — 8/9 stations**) |
| `results/figures/` | `d2_censoring_vs_activity.png` | Daily coverage vs. network extreme-activity scatter, 9 stations |
| `results/gates/` | `censoring_sensitivity_results.md` | D2b: G1/G2/G3 stability check excluding the 9 stations (**all stable**) |
| `results/gates/` | `f6_anisotropy_model_<height>.md` (`_10m`/`_100m`/`_200m`/`_500m`) | F6 anisotropy model per wind height: coefficients, bootstrap CI, effect-size check |
| `results/gates/` | `f6_anisotropy_bootstrap_<height>.parquet` | Cluster bootstrap draws (by directional pair) of the alignment coefficient, per height |
| `results/gates/` | `f6_decision_<height>.md` | F6 written decision per height |
| `results/gates/` | `f6_anisotropy_comparison.md` | F6 cross-height comparison table + conclusion |
| `results/figures/` | `f6_anisotropy_bins_<height>.png` | Coincidence rate (observed/predicted/null) by wind-alignment bin, per height |
| `results/figures/` | `f6_anisotropy_comparison.png` | Alignment coefficient (γ_align) forest plot across the 4 wind heights |
| `results/gates/` | `f6b_timing_model_<height>.md` | F6b advection timing test per height: correlation, OLS, sign-consistency test |
| `results/gates/` | `f6b_timing_bootstrap_<height>.parquet` | Cluster bootstrap draws (by directional pair) of the Spearman correlation, per height |
| `results/gates/` | `f6b_decision_<height>.md` | F6b written decision per height |
| `results/gates/` | `f6b_timing_comparison.md` | F6b cross-height comparison table + consolidated conclusion (**null at all 4 heights**) |
| `results/figures/` | `f6b_timing_scatter_<height>.png` | Observed vs. physically-expected lag (dist/wind speed) scatter, per height |
| `results/figures/` | `f6b_timing_comparison.png` | Spearman ρ forest plot across the 4 wind heights |
| `results/gates/` | `f6_wind_sources_census.md` | Census of all 8 wind/cloud sources considered (4 tested, 4 discarded for heterogeneous reasons) |
| `data/interim/` | `aggregate_clearsky_index.parquet` | Capacity-weighted network k_agg(t) (F7a) |
| `data/interim/` | `aggregate_ramps.parquet` | SDT ramp events detected on k_agg(t), train/test split (F7a) |
| `results/figures/` | `f7a_aggregate_series_example.png` | Example window: individual stations vs. k_agg(t) with detected ramps |
| `results/gates/` | `f7_return_level_fit.parquet` | Aggregate POT/GPD fit (u, ξ̂, σ̂) + return levels by T (Gate G4) |
| `results/gates/` | `f7_return_level_bootstrap.parquet` | Full moving block bootstrap ensemble of return levels (reused by F8's MC ratio CI) |
| `results/gates/` | `f7_backtest_coverage.parquet` | 2017 holdout Poisson exceedance-count backtest by horizon |
| `results/gates/` | `gate4_decision.md` | Gate G4 written decision (**G4 APPROVED**, 100% of horizons covered) |
| `results/figures/` | `f7_return_level_curve.png` | Return level curve with block-bootstrap CI band |
| `results/figures/` | `f7_backtest_coverage.png` | Observed vs. expected exceedance counts + Poisson PI, by horizon |
| `results/gates/` | `f8_independence_bootstrap.parquet` | Independence-scenario return level ensemble (per-station circular day-block shifts) |
| `results/gates/` | `f8_model_implied_bootstrap.parquet` | Model-implied-scenario return level ensemble (F5 two-stage perturbation of real events) |
| `results/gates/` | `f8_rq3_ratio.parquet` | Real/independence/model-implied return levels + MC ratio CI, by horizon |
| `results/gates/` | `f8_rq3_decision.md` | F8 written decision (**RQ3 CONFIRMED**, independence underestimates reserve by 139% at T=1yr) |
| `results/figures/` | `f8_reserve_comparison.png` | 3-scenario return level curves + ratio panel (central RQ3 figure) |
| `results/gates/` | `s10_subsampling_bootstrap.parquet` | Subsampling bootstrap: G1 fraction and L̂ by (k, replicate) |
| `results/gates/` | `s10_subsampling_decision.md` | S10 written decision (**STABLE** — network density robustness) |
| `results/gates/` | `s11_statistical_details.csv` | S11 long-format CSV: Stage 1 inference, G1 p-value percentiles, system counts |
| `results/figures/` | `s10_subsampling_stability.pdf` | G1 fraction and L̂ vs. retained station count (Appendix A.6) |
| `paper/figures/` | `s10_subsampling_stability.pdf` | Copy of S10 figure for LaTeX |
| `paper/figures/` | `fig41_chi_decay.pdf` … `fig47_validation_composite.pdf` | Section 4 figures (P_fig4* scripts) |
| `paper/figures/` | `figA1_sdt_sensitivity.pdf` … `figA4_censoring.pdf`, `s10_subsampling_stability.pdf` | Appendix A figures (P_figA* + S10) |
| `paper/figures/` | `figB1_advection_timing.pdf`, `figB2_spatial_coherence.pdf` | Appendix B figures (P_figB* scripts) |
| `paper/tables/` | `tab41_gate1.tex`, `tab42_model_params.tex`, `tab43_return_levels.tex` | Section 4 LaTeX tables |
| `paper/tables/` | `tabA1_data_quality.tex`, `tabA2_gpd_params.tex` | Appendix A LaTeX tables |
| `paper/` | `RESULTS_STORYBOARD.md` | Storytelling plan for Section 4 (figures, tables, appendices) |

> **Paper artifacts:** run **S10** (network subsampling robustness), **S11** (statistical details CSV), then **Section 10** (`P_*` scripts) after all analysis gates are complete.
> Reference: `paper/RESULTS_STORYBOARD.md`.

> **Next:** F5/Gate G3 is complete and **approved** — the two-stage model (Stage 1:
> excess-coincidence logistic regression with a directional null-probability offset;
> Stage 2: Heffernan–Tawn on the matched-only subset, α(dist) exponential with L=3.72km,
> α₀=0.564 at d→0 — i.e. α≈0.24 at the median matched distance, a modest coupling, since
> α is a regression slope, not a χ) cleanly resolves the pilot's zero-floor artifact and
> is coherent with Gate G1's χ̂ (Spearman ρ=0.32 between per-pair diagnostics). **F6 and F6b
> are now fully complete**, re-run automatically across all 4 available wind heights (KNMI
> 10m surface + CERRA 100/200/500m, once B7b's download finished successfully — ERA5 was
> evaluated and rejected for this purpose, since 31km resolution covers the whole Utrecht
> network in only ~1.4x1.6 grid cells). **F6** (direction/alignment only) gave a result
> that is NOT stable with height: trivial at 10m/100m, but statistically AND practically
> relevant at 200m/500m (up to -8.1% relative change) — with a **sign inverted** relative
> to simple advection (coincidence higher when wind blows against the pair), flagged as
> ambiguous/likely seasonal confounding rather than genuine anisotropy. **F6b** (the
> stronger, more direct timing test: does the observed lag match dist/real-wind-speed?)
> gave a **null result at ALL 4 heights** (|ρ|≤0.046, well below the 0.10 relevance
> threshold), including at 500m where F6's effect was strongest — this is the test that
> actually decides, and it does not support a physically trackable advection mechanism at
> any height. **Consolidated conclusion:** the Gate G1 tail dependence remains best
> explained as shared regional weather regime, not individually-advected cloud shadows;
> this strengthens (rather than undermines) the original choice to model dependence via
> copulas/Heffernan-Tawn rather than an explicit transport mechanism. The Gate G2 ξ̂
> cross-check is resolved; the 5 "up"-direction series with genuine tail-boundedness should
> be reported as a finding, not silently dropped, in the paper's marginal-modelling
> section.
>
> **F7 and F8 are now complete, closing the pilot.** F7a defined the network as a single
> capacity-weighted virtual plant and re-used B5's unchanged SDT detector on k_agg(t). F7
> fitted a POT/GPD to this aggregate series with the same adaptive-threshold +
> Ferro-Segers-declustering methodology as Gate G2, and moving-block-bootstrap CIs reusing
> Gate G1's block-length estimator — **Gate G4 APPROVED**, with 100% of backtested horizons
> (4/4) covered against the 2017 holdout year via a Poisson exceedance-count test. F8
> compared three scenarios under the identical POT/GPD methodology: REAL (observed),
> INDEPENDENCE (empirical counterfactual — per-station circular day-block shifts, zero
> dependence assumptions), and MODEL-IMPLIED (real events with neighbor responses replaced
> by Gate G3's two-stage Heffernan-Tawn model). **RQ3 CONFIRMED:** the real/independence
> return-level ratio at T=1 year is 2.388 (95% CI [2.033, 2.698], excludes 1) — i.e., sizing
> reserve capacity under the naive independence assumption underestimates the true
> requirement by 139% (95% CI [103%, 170%]). The model-implied scenario tracks the real one
> within a median 4.5% deviation across horizons, validating that Gate G3's
> conditional-extremes model has genuine predictive value for aggregate portfolio risk, not
> just pairwise post-hoc fit. **All three RQs are now answered and all four gates (G1–G4)
> are approved — the pilot is complete.** Remaining optional work before paper writing:
> `B6_sensitivity.py` (ε/Δ SDT sensitivity grid, already run — see
> `data/interim/sensitivity_grid.csv`) and a final end-to-end consistency pass across all
> `results/gates/*_decision.md` files.